# Hybrid Recommender

This notebook trains and evaluates:
- Collaborative Filtering (CF)
- Content-Based Filtering (CBF) from restaurant categories
- A per-user selector that picks CF or CBF

It is scoped to one city (default `Philadelphia`) for tractability.

**Outputs:**
- End-to-end recommendation workflow with sectioned outputs from data prep to explanation tables.

**What it tells us:**
- Whether the hybrid recommender can rank useful restaurants and explain why those items were selected.


## 1) Setup and experiment configuration

This section imports dependencies and defines the global configuration for data scope,
model behavior, and reproducibility.

**Outputs:**
- Printed paths and selected city (`Data dir`, `Target city`).
- The effective experiment settings (`K`, thresholds, sampling fraction).

**What it tells us:**
- Confirms run context before expensive processing starts.
- Makes the experiment reproducible and easy to compare across runs.

In [58]:
from __future__ import annotations

import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# Config
# -------------------------
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "original_data" / "yelp_json"
BUSINESS_PATH = DATA_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = DATA_DIR / "yelp_academic_dataset_review.json"

# City for which recommendation is generated.
TARGET_CITY = "Philadelphia"
# K controls how many restaurants are returned/evaluated.
K = 10
# MIN_* thresholds remove very sparse users/items before modeling.
MIN_USER_INTERACTIONS = 3
MIN_BUSINESS_INTERACTIONS = 5
# SELECTOR_THRESHOLD_T decides when to trust CBF for sparse profiles.
SELECTOR_THRESHOLD_T = 3
# LIKE_STARS is the cutoff for "positive" interactions.
LIKE_STARS = 4.0
SEED = 42
# SAMPLE_USER_FRAC keeps experiments lightweight during iteration. Set to 1.0 for full data.
SAMPLE_USER_FRAC = 0.20  # 20% of users for faster experiments



np.random.seed(SEED)

print(f"Data dir: {DATA_DIR}")
print(f"Target city: {TARGET_CITY}")


Data dir: C:\Users\kello\PycharmProjects\ISA-Team-8\original_data\yelp_json
Target city: Philadelphia


## 2) Data loading and preprocessing helpers

This section defines helper functions for streaming Yelp JSON files, parsing
categories, and applying interaction denoising/sampling.

**Outputs:**
- No direct table output; this cell defines reusable helper functions.

**What it tells us:**
- Documents how raw JSON is transformed into model-ready tables.
- Establishes filtering rules used later in evaluation and serving.

In [59]:
def iter_json_lines(path: Path):
    """Yield one JSON object per line from a Yelp JSONL file."""
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


def parse_categories(raw: str | None) -> list[str]:
    """Split category text into a deduplicated, lowercase list."""
    if not raw:
        return []
    cats = [c.strip().lower() for c in raw.split(",") if c and c.strip()]
    return sorted(set(cats))


def load_city_restaurants(business_path: Path, city: str) -> pd.DataFrame:
    """Load only restaurant businesses from the selected city."""
    records = []
    for obj in iter_json_lines(business_path):
        if str(obj.get("city", "")).strip() != city:
            continue

        cats = parse_categories(obj.get("categories"))
        if not cats:
            continue
        if not any("restaurant" in c for c in cats):
            continue

        records.append({
            "business_id": obj.get("business_id"),
            "name": obj.get("name"),
            "city": obj.get("city"),
            "state": obj.get("state"),
            "stars": float(obj.get("stars", 0.0)),
            "review_count": int(obj.get("review_count", 0)),
            "is_open": int(obj.get("is_open", 1)),
            "category_list": cats,
        })

    return pd.DataFrame(records)


def load_reviews_for_businesses(review_path: Path, business_ids: set[str]) -> pd.DataFrame:
    """Load reviews only for businesses that passed city/category filtering."""
    rows = []
    for obj in iter_json_lines(review_path):
        bid = obj.get("business_id")
        if bid not in business_ids:
            continue
        rows.append({
            "review_id": obj.get("review_id"),
            "user_id": obj.get("user_id"),
            "business_id": bid,
            "stars": float(obj.get("stars", 0.0)),
            "date": obj.get("date"),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])
    return df


def sample_users_by_fraction(review_df: pd.DataFrame, frac: float, seed: int) -> pd.DataFrame:
    """Randomly sample users and keep all of their reviews for faster runs."""
    if frac >= 1.0:
        return review_df
    users = review_df["user_id"].drop_duplicates().to_numpy()
    if users.size == 0:
        return review_df

    target_n = max(1, int(len(users) * frac))
    rng = np.random.default_rng(seed)
    sampled_users = set(rng.choice(users, size=target_n, replace=False).tolist())
    return review_df[review_df["user_id"].isin(sampled_users)].reset_index(drop=True)


def apply_interaction_filters(review_df: pd.DataFrame, min_u: int, min_b: int) -> pd.DataFrame:
    """Iteratively remove users/items below interaction thresholds."""
    df = review_df.copy()
    # Two passes are usually enough for this denoising step.
    for _ in range(2):
        user_ok = df["user_id"].value_counts()
        user_ok = set(user_ok[user_ok >= min_u].index)
        df = df[df["user_id"].isin(user_ok)]

        biz_ok = df["business_id"].value_counts()
        biz_ok = set(biz_ok[biz_ok >= min_b].index)
        df = df[df["business_id"].isin(biz_ok)]

    return df.reset_index(drop=True)


## 3) Build filtered dataset and temporal split

This section loads city-scoped restaurant data, samples users for faster iteration,
and creates train/validation/test splits by time.

**Outputs:**
- Printed counts for businesses, reviews, users, and train/val/test sizes.
- Sampling fraction used for this run.

**What it tells us:**
- Shows actual problem size and expected runtime cost.
- Verifies temporal split is populated correctly before modeling.

In [60]:
def temporal_split_leave_last_two(interactions: pd.DataFrame):
    """Per-user temporal split: train + optional val + test from latest records."""
    train_parts, val_parts, test_parts = [], [], []
    cohorts = {}

    for _, g in interactions.sort_values("date").groupby("user_id", sort=False):
        n = len(g)
        if n >= 3:
            train_parts.append(g.iloc[:-2])
            val_parts.append(g.iloc[-2:-1])
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "warm"
        elif n == 2:
            train_parts.append(g.iloc[:1])
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "sparse"
        else:
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "cold"

    def _cat(parts):
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=interactions.columns)

    return _cat(train_parts), _cat(val_parts), _cat(test_parts), cohorts


business_df = load_city_restaurants(BUSINESS_PATH, TARGET_CITY)
business_ids = set(business_df["business_id"].tolist())
review_df = load_reviews_for_businesses(REVIEW_PATH, business_ids)
# First denoising pass removes extreme sparsity before optional sampling.
review_df = apply_interaction_filters(review_df, MIN_USER_INTERACTIONS, MIN_BUSINESS_INTERACTIONS)
review_df = sample_users_by_fraction(review_df, SAMPLE_USER_FRAC, SEED)

# Re-apply denoising after sampling to keep matrix quality stable.
review_df = apply_interaction_filters(review_df, min_u=2, min_b=3)

# Keep only businesses that survived interaction denoising.
active_business_ids = set(review_df["business_id"].unique())
business_df = business_df[business_df["business_id"].isin(active_business_ids)].copy()

train_df, val_df, test_df, user_cohort = temporal_split_leave_last_two(review_df)

print("Businesses in scope:", len(business_df))
print("Reviews after filtering:", len(review_df))
print("Users after filtering:", review_df['user_id'].nunique())
print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Sampling fraction:", SAMPLE_USER_FRAC)


Businesses in scope: 4062
Reviews after filtering: 96669
Users after filtering: 10091
Train/Val/Test: 76644 9934 10091
Sampling fraction: 0.2


## 4) Collaborative filtering model (CF)

This section builds the user-item matrix and factorizes it to produce latent factors
used for collaborative ranking.

**Outputs:**
- Printed latent factor shapes (`user_factors`, `item_factors`).

**What it tells us:**
- Confirms CF model was trained and dimensions are valid.
- Helps detect under/over-sized latent space early.

In [61]:
all_users = sorted(review_df["user_id"].unique().tolist())
all_items = sorted(business_df["business_id"].unique().tolist())

u2i = {u: i for i, u in enumerate(all_users)}
b2i = {b: i for i, b in enumerate(all_items)}
i2u = {i: u for u, i in u2i.items()}
i2b = {i: b for b, i in b2i.items()}

def build_interaction_matrix(df: pd.DataFrame, user_map: dict[str, int], item_map: dict[str, int]) -> csr_matrix:
    """Convert interactions into a sparse user-item matrix for CF."""
    if df.empty:
        return csr_matrix((len(user_map), len(item_map)), dtype=np.float32)

    rows = df["user_id"].map(user_map).to_numpy()
    cols = df["business_id"].map(item_map).to_numpy()
    vals = df["stars"].astype(np.float32).to_numpy()
    return csr_matrix((vals, (rows, cols)), shape=(len(user_map), len(item_map)), dtype=np.float32)


R_train = build_interaction_matrix(train_df, u2i, b2i)

# Keep latent dimension bounded so SVD remains stable on sampled matrices.
n_components = int(min(64, max(2, min(R_train.shape) - 1)))
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
user_factors = svd.fit_transform(R_train)
item_factors = svd.components_.T

print("CF factors:", user_factors.shape, item_factors.shape)


CF factors: (10091, 64) (4062, 64)


## 5) Content-based model (CBF) from categories

This section builds item category vectors and user preference profiles from training
interactions.

**Outputs:**
- Printed item-category matrix shape and vocabulary size.
- In-memory user profile vectors (`user_profiles`).

**What it tells us:**
- Confirms category feature space quality and scale.
- Indicates whether CBF has enough signal to personalize.

In [62]:
biz_cat = business_df[["business_id", "category_list"]].copy()
# Deduplicate only by business_id; category_list contains lists (unhashable for full-row dedupe).
biz_cat = biz_cat.drop_duplicates(subset=["business_id"], keep="first").set_index("business_id")
ordered_categories = [biz_cat["category_list"].get(b, []) for b in all_items]

mlb = MultiLabelBinarizer()
X_item_cat = mlb.fit_transform(ordered_categories).astype(np.float32)
item_cat_norm = np.linalg.norm(X_item_cat, axis=1) + 1e-12

train_user_groups = train_df.groupby("user_id")
user_profiles = {}
# Build one category-profile per user from training interactions.
# Positive interactions are preferred; fallback uses all interactions.
for uid in all_users:
    if uid not in train_user_groups.groups:
        user_profiles[uid] = None
        continue

    g = train_user_groups.get_group(uid)
    liked = g[g["stars"] >= LIKE_STARS]
    src = liked if not liked.empty else g

    idxs = [b2i[b] for b in src["business_id"].tolist() if b in b2i]
    if not idxs:
        user_profiles[uid] = None
        continue

    w = src["stars"].astype(np.float32).to_numpy()
    if len(w) != len(idxs):
        w = np.ones(len(idxs), dtype=np.float32)

    mat = X_item_cat[idxs]
    # Weighted average gives higher-rated restaurants more impact on the profile.
    profile = np.average(mat, axis=0, weights=w)
    norm = np.linalg.norm(profile) + 1e-12
    user_profiles[uid] = profile / norm

print("CBF item matrix:", X_item_cat.shape, "vocab size:", len(mlb.classes_))


CBF item matrix: (4062, 335) vocab size: 335


## 6) Recommendation functions

This section defines ranking functions for CF, CBF, and popularity fallback.
For offline evaluation we exclude only training interactions.
For user-facing recommendations we exclude all known visited restaurants.

**Outputs:**
- No direct table output; this cell defines recommendation behavior.

**What it tells us:**
- Separates evaluation-time logic from serving-time non-repeat policy.
- Ensures deployed recommendations do not include previously visited places.

In [63]:
train_seen = defaultdict(set)
for r in train_df.itertuples(index=False):
    train_seen[r.user_id].add(r.business_id)

# Full-history visited set used for serving-time non-repeat recommendations.
all_seen = defaultdict(set)
for r in review_df.itertuples(index=False):
    all_seen[r.user_id].add(r.business_id)

popularity_rank = train_df.groupby("business_id")["stars"].mean().sort_values(ascending=False).index.tolist()

def top_k_from_scores(scores: np.ndarray, seen: set[str], k: int) -> list[str]:
    """Return top-k item ids after masking already-seen businesses."""
    if scores.size == 0:
        return []

    work = scores.copy()
    if seen:
        # Mask seen items so they cannot appear in recommendations.
        seen_idx = [b2i[b] for b in seen if b in b2i]
        work[seen_idx] = -np.inf

    k = int(min(k, len(work)))
    if k <= 0:
        return []

    cand = np.argpartition(work, -k)[-k:]
    # `argpartition` is faster than full sorting for top-k extraction.
    cand = cand[np.argsort(work[cand])[::-1]]
    return [i2b[int(i)] for i in cand if np.isfinite(work[int(i)])]


def recommend_cf(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """CF ranking from latent factors with popularity fallback."""
    if uid not in u2i:
        return popularity_rank[:k]

    uidx = u2i[uid]
    scores = user_factors[uidx] @ item_factors.T
    seen = train_seen.get(uid, set()) if seen_override is None else seen_override
    recs = top_k_from_scores(scores, seen, k)
    return recs if recs else popularity_rank[:k]


def recommend_cbf(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """CBF ranking from category-profile similarity with fallback."""
    profile = user_profiles.get(uid)
    if profile is None:
        return popularity_rank[:k]

    scores = (X_item_cat @ profile) / item_cat_norm
    seen = train_seen.get(uid, set()) if seen_override is None else seen_override
    recs = top_k_from_scores(scores, seen, k)
    return recs if recs else popularity_rank[:k]


def recommend_hybrid(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """Use selector to route user to CF, CBF, or popularity."""
    model = choose_model_for_user(uid)
    if model == "cf":
        return recommend_cf(uid, k, seen_override=seen_override)
    if model == "cbf":
        return recommend_cbf(uid, k, seen_override=seen_override)

    # Popularity fallback with optional seen exclusion.
    if seen_override:
        return [b for b in popularity_rank if b not in seen_override][:k]
    return popularity_rank[:k]


def recommend_hybrid_serving(uid: str, k: int = K) -> list[str]:
    """Serving wrapper: never return already visited restaurants."""
    # Serving policy: never recommend restaurants already visited by this user.
    return recommend_hybrid(uid, k, seen_override=all_seen.get(uid, set()))


## 7) Metrics and model-selection logic

This section defines ranking metrics and per-user model routing (CF vs CBF)
based on validation performance and interaction thresholds.

**Outputs:**
- No direct DataFrame; defines metric functions and selector maps.
- Creates `selector_choice`, `val_rel`, `test_rel`, and `train_counts`.

**What it tells us:**
- Encodes how model quality is measured.
- Encodes why a user is routed to CF, CBF, or popularity fallback.

In [64]:
def dcg_at_k(binary_rel: list[int], k: int) -> float:
    """Discounted cumulative gain for a binary relevance list."""
    vals = np.array(binary_rel[:k], dtype=np.float32)
    if vals.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, vals.size + 2))
    return float(np.sum(vals * discounts))


def ndcg_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Normalized DCG at K."""
    if not rel_set:
        return 0.0
    gains = [1 if r in rel_set else 0 for r in recs[:k]]
    dcg = dcg_at_k(gains, k)
    ideal = [1] * min(len(rel_set), k)
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else dcg / idcg


def precision_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Precision at K."""
    if k <= 0:
        return 0.0
    hit = sum(1 for r in recs[:k] if r in rel_set)
    return float(hit / k)


def recall_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Recall at K."""
    if not rel_set:
        return 0.0
    hit = sum(1 for r in recs[:k] if r in rel_set)
    return float(hit / len(rel_set))


def hit_rate_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Whether at least one relevant item appears in top-K."""
    return 1.0 if any(r in rel_set for r in recs[:k]) else 0.0


def ap_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Average precision at K."""
    if not rel_set:
        return 0.0
    hit_count = 0
    score = 0.0
    for i, r in enumerate(recs[:k], start=1):
        if r in rel_set:
            hit_count += 1
            score += hit_count / i
    denom = min(len(rel_set), k)
    return score / denom if denom else 0.0


def build_relevance_by_user(df: pd.DataFrame, like_stars: float = LIKE_STARS) -> dict[str, set[str]]:
    """Build user->relevant-business set using a star cutoff."""
    if df.empty:
        return {}
    rel = defaultdict(set)
    for r in df.itertuples(index=False):
        if r.stars >= like_stars:
            rel[r.user_id].add(r.business_id)
    return dict(rel)


val_rel = build_relevance_by_user(val_df)
test_rel = build_relevance_by_user(test_df)
train_counts = train_df["user_id"].value_counts().to_dict()

# Per-user selector from validation.
selector_choice = {}
# Keep which model (CF/CBF) wins on validation ranking for each user.
for uid, rel_set in val_rel.items():
    cf_score = ndcg_at_k(recommend_cf(uid, K), rel_set, K)
    cbf_score = ndcg_at_k(recommend_cbf(uid, K), rel_set, K)
    selector_choice[uid] = "cf" if cf_score >= cbf_score else "cbf"


def choose_model_for_user(uid: str) -> str:
    """Route user to model based on history size and validation winner."""
    n_train = train_counts.get(uid, 0)
    if n_train == 0:
        return "pop"
    if n_train < SELECTOR_THRESHOLD_T:
        return "cbf"
    return selector_choice.get(uid, "cf")


# `recommend_hybrid` is defined in section 6 with optional seen overrides.


## 8) Overall and cohort evaluation

This section compares Popularity, CF, CBF, and Hybrid on overall metrics and on
warm/sparse/cold cohorts.

**How to read the metrics (higher is better):**
- `Precision@10`: fraction of top-10 recommendations that are relevant.
- `Recall@10`: fraction of a user's relevant items recovered in top-10.
- `HitRate@10`: percent of users with at least one relevant item in top-10.
- `NDCG@10`: ranking quality that rewards putting relevant items near the top.
- `MAP@10`: average precision across top-10 ranking positions.
- `Coverage@10`: share of catalog that appears in recommendations (diversity/novelty proxy).

**What cohorts mean:**
- `warm`: users with enough history (>=3 interactions in this split).
- `sparse`: users with very limited history (exactly 2 interactions here).
- `cold`: users with no train history (not enough data for personalized training).

**About low-looking values:**
On large, sparse implicit recommendation tasks, absolute values can look small.
The key signal is relative comparison between models and cohorts, not raw magnitude alone.

**Outputs:**
- `overall_df`: model comparison on all evaluable users.
- `cohort_df`: model comparison split by `warm`, `sparse`, `cold` users.
- `cohort_user_counts` and optional low-support warning table.

**What it tells us:**
- Identifies the best model overall and per cohort.
- Shows whether results are reliable (or unstable) based on cohort support.

In [65]:
def evaluate_model(model_name: str, rec_fn, rel_by_user: dict[str, set[str]], k: int = K) -> dict:
    """Evaluate one recommender function on a relevance dictionary."""
    users = [u for u in rel_by_user.keys()]
    if not users:
        return {"model": model_name, "users": 0}

    p_list, r_list, h_list, n_list, ap_list = [], [], [], [], []
    recommended_catalog = set()

    for uid in users:
        recs = rec_fn(uid, k)
        rel = rel_by_user.get(uid, set())

        p_list.append(precision_at_k(recs, rel, k))
        r_list.append(recall_at_k(recs, rel, k))
        h_list.append(hit_rate_at_k(recs, rel, k))
        n_list.append(ndcg_at_k(recs, rel, k))
        ap_list.append(ap_at_k(recs, rel, k))
        recommended_catalog.update(recs[:k])

    return {
        "model": model_name,
        "users": len(users),
        f"Precision@{k}": float(np.mean(p_list)),
        f"Recall@{k}": float(np.mean(r_list)),
        f"HitRate@{k}": float(np.mean(h_list)),
        f"NDCG@{k}": float(np.mean(n_list)),
        f"MAP@{k}": float(np.mean(ap_list)),
        f"Coverage@{k}": float(len(recommended_catalog) / max(1, len(all_items))),
    }


def recommend_pop(uid: str, k: int = K) -> list[str]:
    """Popularity baseline for comparison and fallback."""
    return popularity_rank[:k]


results = [
    # Overall comparison: every model evaluated on the same test relevance set.
    evaluate_model("Popularity", recommend_pop, test_rel, K),
    evaluate_model("CF", recommend_cf, test_rel, K),
    evaluate_model("CBF", recommend_cbf, test_rel, K),
    evaluate_model("Hybrid", recommend_hybrid, test_rel, K),
]

overall_df = pd.DataFrame(results).sort_values(by=f"NDCG@{K}", ascending=False).reset_index(drop=True)
overall_df

# Cohort-level breakdown helps show where hybrid helps (typically sparse users).
def filter_rel_by_users(rel_by_user: dict[str, set[str]], users: set[str]) -> dict[str, set[str]]:
    """Restrict relevance mapping to a target user subset."""
    return {u: rel_by_user[u] for u in rel_by_user if u in users}

warm_users = {u for u, c in user_cohort.items() if c == "warm"}
sparse_users = {u for u, c in user_cohort.items() if c == "sparse"}
cold_users = {u for u, c in user_cohort.items() if c == "cold"}

cohort_rows = []
# Cohort-level view highlights how each model behaves with different history sizes.
for cohort_name, cohort_users in [("warm", warm_users), ("sparse", sparse_users), ("cold", cold_users)]:
    rel_subset = filter_rel_by_users(test_rel, cohort_users)
    cohort_rows.extend([
        {"cohort": cohort_name, **evaluate_model("Popularity", recommend_pop, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("CF", recommend_cf, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("CBF", recommend_cbf, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("Hybrid", recommend_hybrid, rel_subset, K)},
    ])

cohort_df = pd.DataFrame(cohort_rows)
cohort_df.sort_values(["cohort", f"NDCG@{K}"], ascending=[True, False]).reset_index(drop=True)

# Quick cohort support check: useful when NaNs appear from tiny/empty cohorts.
cohort_user_counts = pd.Series(user_cohort).value_counts().rename_axis("cohort").reset_index(name="user_count")
display(cohort_user_counts)

low_support = cohort_user_counts[cohort_user_counts["user_count"] < 30]
if not low_support.empty:
    print("Warning: some cohorts have low support (<30 users), metrics can be unstable.")
    display(low_support)


,cohort,user_count
0,warm,9934
1,sparse,157


## 9) Basic recommendation preview

This section returns a simple recommendation table for a sample user.
These recommendations use serving-time exclusions (no already visited restaurants).

**Outputs:**
- `sample_users`: a few user ids from the test relevance set.
- A compact top-K recommendation table (`rank`, `selected_model`, business info).

**What it tells us:**
- Quick sanity check that end-to-end recommendation runs.
- Human-readable preview before deeper confidence analysis.

In [66]:
def recommendation_table(uid: str, k: int = 10) -> pd.DataFrame:
    """Simple serving-time recommendation table for a user."""
    model = choose_model_for_user(uid)
    recs = recommend_hybrid_serving(uid, k)

    out = business_df[business_df["business_id"].isin(recs)][["business_id", "name", "city", "stars", "category_list"]].copy()
    out["rank"] = out["business_id"].map({b: i + 1 for i, b in enumerate(recs)})
    out["selected_model"] = model
    out = out.sort_values("rank")
    return out[["rank", "selected_model", "business_id", "name", "city", "stars", "category_list"]]


sample_users = list(test_rel.keys())[:3]
sample_users


['H4JNrBAoyCk_ZMZWbAf8OA', 'PO-U11FmTDiqCEqtilFjVQ', 'GitE04dW0jYxH5LplUTDBA']

## 10) Confidence report for each recommendation

This section explains model confidence at two levels:
- **model selection confidence**: how strongly the system prefers CF/CBF/popularity
- **item confidence**: relative strength of each restaurant score within the top-K list

The confidence table also applies serving-time non-repeat filtering.

**Outputs:**
- `recommendation_confidence_table(...)` with per-item confidence columns:
  - `item_confidence`, `raw_score`, `model_confidence`, `model_reason`, `selection_reason`, `already_visited`.
- A rendered confidence table for one sample user.

**What it tells us (most important):**
- How strongly the selected model appears supported for that user (`model_confidence`).
- Which recommended restaurants are stronger/weaker within the returned list (`item_confidence`).
- Why the model was selected and what scoring logic produced the final ranking.
- Whether serving constraints are respected (`already_visited` should be `False`).

In [67]:
popularity_score_map = train_df.groupby("business_id")["stars"].mean().to_dict()


def _safe_minmax(values: np.ndarray) -> np.ndarray:
    """Normalize values to [0,1] with safe handling of constant vectors."""
    if values.size == 0:
        return values
    vmin = float(np.min(values))
    vmax = float(np.max(values))
    if vmax - vmin < 1e-12:
        return np.full(values.shape, 0.5, dtype=np.float32)
    return ((values - vmin) / (vmax - vmin)).astype(np.float32)


def model_selection_confidence(uid: str) -> tuple[float, str]:
    """Heuristic confidence score for the selected recommendation model."""
    n_train = train_counts.get(uid, 0)
    selected = choose_model_for_user(uid)

    if n_train == 0:
        return 0.35, "Cold-start user: fallback to popularity baseline."

    if n_train < SELECTOR_THRESHOLD_T:
        conf = min(0.65, 0.40 + 0.05 * n_train)
        return conf, f"Sparse history ({n_train} interactions): prefer CBF profile matching."

    rel_set = val_rel.get(uid, set())
    if rel_set:
        cf_val = ndcg_at_k(recommend_cf(uid, K), rel_set, K)
        cbf_val = ndcg_at_k(recommend_cbf(uid, K), rel_set, K)
        margin = abs(cf_val - cbf_val)
        conf = min(0.95, 0.55 + 3.0 * margin + min(0.15, n_train / 120.0))
        return conf, f"Validation margin supports {selected.upper()} (|ΔNDCG|={margin:.4f})."

    conf = min(0.85, 0.55 + min(0.25, n_train / 80.0))
    return conf, f"No validation labels; confidence based on history size ({n_train})."


def recommendation_confidence_table(uid: str, k: int = 10) -> pd.DataFrame:
    """Detailed serving-time table with item and model confidence fields."""
    model = choose_model_for_user(uid)
    seen_for_serving = all_seen.get(uid, set())

    if model == "cf" and uid in u2i:
        scores = user_factors[u2i[uid]] @ item_factors.T
        model_reason = "CF confidence uses similar-user latent preference strength."
    elif model == "cbf" and user_profiles.get(uid) is not None:
        scores = (X_item_cat @ user_profiles[uid]) / item_cat_norm
        model_reason = "CBF confidence uses category-profile similarity."
    else:
        scores = np.array([popularity_score_map.get(b, 0.0) for b in all_items], dtype=np.float32)
        model_reason = "Popularity confidence uses average city rating in train data."

    recs = top_k_from_scores(scores, seen_for_serving, k)
    # Confidence is relative to this user's returned top-k list.
    rec_scores = np.array([float(scores[b2i[b]]) for b in recs], dtype=np.float32)
    rec_conf = _safe_minmax(rec_scores)

    model_conf, selection_reason = model_selection_confidence(uid)

    base = business_df[business_df["business_id"].isin(recs)][
        ["business_id", "name", "city", "stars", "category_list"]
    ].copy()
    base["rank"] = base["business_id"].map({b: i + 1 for i, b in enumerate(recs)})
    base["raw_score"] = base["business_id"].map({b: s for b, s in zip(recs, rec_scores)})
    base["item_confidence"] = base["business_id"].map({b: c for b, c in zip(recs, rec_conf)})
    base["selected_model"] = model
    base["model_confidence"] = float(model_conf)
    base["model_reason"] = model_reason
    base["selection_reason"] = selection_reason
    base["already_visited"] = base["business_id"].isin(seen_for_serving)

    base = base.sort_values("rank")
    return base[[
        "rank",
        "selected_model",
        "model_confidence",
        "item_confidence",
        "raw_score",
        "already_visited",
        "business_id",
        "name",
        "city",
        "stars",
        "category_list",
        "model_reason",
        "selection_reason",
    ]]


if sample_users:
    display(recommendation_confidence_table(sample_users[0], k=10))
else:
    print("No evaluable users found in test relevance set.")


,rank,selected_model,model_confidence,item_confidence,raw_score,already_visited,business_id,name,city,stars,category_list,model_reason,selection_reason
2442,1,cf,0.658333,1.000000,0.745409,False,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,4.0,"[american (new), breakfast & brunch, french, i...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
2044,2,cf,0.658333,0.695346,0.659648,False,wbDRmtxaKRpBOjutvV6TEA,Barclay Prime,Philadelphia,4.5,"[restaurants, steakhouses]",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
2788,3,cf,0.658333,0.392770,0.574473,False,R17gwW6zn9ilslbdvKdgsg,Alma de Cuba,Philadelphia,4.0,"[bars, caribbean, cocktail bars, cuban, latin ...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
2135,4,cf,0.658333,0.292057,0.546122,False,vpNJJNVgVmf2u1lfdFh81w,Matyson,Philadelphia,4.0,"[american (new), breakfast & brunch, restaurants]",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
2501,5,cf,0.658333,0.220756,0.526051,False,Qw7tz-UkPrpXaVidWuab4Q,Philadelphia Museum of Art,Philadelphia,4.5,"[american (new), art galleries, art museums, a...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
1634,6,cf,0.658333,0.190042,0.517405,False,sb8-TzsXOV7IsErbpHZo3g,Positano Coast by Aldo Lamberti,Philadelphia,3.5,"[bars, breakfast & brunch, italian, lounges, n...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
343,7,cf,0.658333,0.161668,0.509417,False,d_tRshM-w6S4QxE4VVi8tQ,Jones,Philadelphia,3.5,"[american (new), american (traditional), diner...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
5378,8,cf,0.658333,0.142765,0.504096,False,OdIBX09glfXNVSyd0RnIeg,Monk's Cafe,Philadelphia,4.0,"[bars, belgian, gastropubs, nightlife, pubs, r...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
21,9,cf,0.658333,0.011269,0.467080,False,7mpYTDb24SywNMRn3yeakQ,The Twisted Tail,Philadelphia,4.0,"[american (new), american (traditional), bars,...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).
1016,10,cf,0.658333,0.000000,0.463907,False,6ewV-e7-39oqYUq3yZuIyw,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).


## 11) Compare recommendations to user positive history

This section helps interpretation by comparing recommended restaurants to places
the user rated positively in the past (using category overlap).

**Outputs:**
- `rec_compare_df`: recommendation table augmented with:
  - `liked_category_overlap`, `liked_category_overlap_ratio`.
- `liked_history_df`: user's positively rated restaurants sorted by stars.

**What it tells us (most important):**
- How aligned each recommendation is with the user's known likes.
- Whether the system is discovering related categories or drifting away from preferences.
- Provides an explanation layer beyond ranking metrics for stakeholder review.

In [68]:
# Build positive history from full filtered data so explanation tables have enough context.
positive_history = build_relevance_by_user(review_df, like_stars=LIKE_STARS)


def compare_recommendations_to_positive_history(uid: str, k: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare recommendations to user's liked-history via category overlap."""
    rec_df = recommendation_confidence_table(uid, k=k).copy()
    liked_ids = positive_history.get(uid, set())

    liked_df = business_df[business_df["business_id"].isin(liked_ids)][
        ["business_id", "name", "city", "stars", "category_list"]
    ].copy()

    # Collapse liked restaurants into one set of preferred categories.
    liked_categories = set()
    if not liked_df.empty:
        for cats in liked_df["category_list"]:
            liked_categories.update(cats)

    def overlap_count(cats: list[str]) -> int:
        """Count overlapping categories with the user's positively rated history."""
        return len(set(cats).intersection(liked_categories))

    rec_df["liked_category_overlap"] = rec_df["category_list"].apply(overlap_count)
    # Ratio normalizes overlap by recommendation category count.
    rec_df["liked_category_overlap_ratio"] = rec_df["liked_category_overlap"] / rec_df["category_list"].apply(lambda x: max(1, len(x)))

    liked_df = liked_df.sort_values("stars", ascending=False)
    return rec_df, liked_df


if sample_users:
    rec_compare_df, liked_history_df = compare_recommendations_to_positive_history(sample_users[0], k=10)
    display(rec_compare_df)
    display(liked_history_df.head(15))
else:
    print("No evaluable users found in test relevance set.")


,rank,selected_model,model_confidence,item_confidence,raw_score,already_visited,business_id,name,city,stars,category_list,model_reason,selection_reason,liked_category_overlap,liked_category_overlap_ratio
2442,1,cf,0.658333,1.000000,0.745409,False,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,4.0,"[american (new), breakfast & brunch, french, i...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,3,0.500000
2044,2,cf,0.658333,0.695346,0.659648,False,wbDRmtxaKRpBOjutvV6TEA,Barclay Prime,Philadelphia,4.5,"[restaurants, steakhouses]",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,1,0.500000
2788,3,cf,0.658333,0.392770,0.574473,False,R17gwW6zn9ilslbdvKdgsg,Alma de Cuba,Philadelphia,4.0,"[bars, caribbean, cocktail bars, cuban, latin ...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,6,0.857143
2135,4,cf,0.658333,0.292057,0.546122,False,vpNJJNVgVmf2u1lfdFh81w,Matyson,Philadelphia,4.0,"[american (new), breakfast & brunch, restaurants]",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,3,1.000000
2501,5,cf,0.658333,0.220756,0.526051,False,Qw7tz-UkPrpXaVidWuab4Q,Philadelphia Museum of Art,Philadelphia,4.5,"[american (new), art galleries, art museums, a...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,4,0.400000
1634,6,cf,0.658333,0.190042,0.517405,False,sb8-TzsXOV7IsErbpHZo3g,Positano Coast by Aldo Lamberti,Philadelphia,3.5,"[bars, breakfast & brunch, italian, lounges, n...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,4,0.500000
343,7,cf,0.658333,0.161668,0.509417,False,d_tRshM-w6S4QxE4VVi8tQ,Jones,Philadelphia,3.5,"[american (new), american (traditional), diner...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,4,0.800000
5378,8,cf,0.658333,0.142765,0.504096,False,OdIBX09glfXNVSyd0RnIeg,Monk's Cafe,Philadelphia,4.0,"[bars, belgian, gastropubs, nightlife, pubs, r...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,4,0.666667
21,9,cf,0.658333,0.011269,0.467080,False,7mpYTDb24SywNMRn3yeakQ,The Twisted Tail,Philadelphia,4.0,"[american (new), american (traditional), bars,...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,6,0.750000
1016,10,cf,0.658333,0.000000,0.463907,False,6ewV-e7-39oqYUq3yZuIyw,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break...",CF confidence uses similar-user latent prefere...,Validation margin supports CF (|ΔNDCG|=0.0000).,6,0.857143


,business_id,name,city,stars,category_list
3487,6_T2xzR74JqGCTPefAD8Tw,Morimoto,Philadelphia,4.5,"[american (new), american (traditional), asian..."
4136,K3RURR9lIEE4JjOaPt99zg,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break..."
2231,ZKPrXH_GNW_AtZ31tP3NmA,White Dog Cafe,Philadelphia,4.0,"[american (new), american (traditional), bars,..."
3591,aXr74YWu67vUtWsbmICbag,Rose Tattoo Cafe,Philadelphia,4.0,"[american (new), american (traditional), bars,..."
3699,55gCXlWDDCdttR3yRss1xw,The Rittenhouse Hotel,Philadelphia,4.0,"[event planning & services, hotels, hotels & t..."
5033,nIAbuktMEzVjT4P9pG89rQ,Buddakan,Philadelphia,4.0,"[asian fusion, chinese, restaurants]"
5681,JHRlwxxKY0JJcU97rJ-Bug,Cuba Libre Restaurant & Rum Bar - Philadelphia,Philadelphia,3.5,"[bars, beer, breakfast & brunch, cuban, dance ..."
1969,Co3Ogqy6y2JgZdG0wBlrUQ,Ten Stone Bar & Restaurant,Philadelphia,3.0,"[american (new), asian fusion, bars, nightlife..."
5253,bAqDJlYqB9j3tMVjhp29EQ,Mezze,Philadelphia,3.0,"[food, greek, grocery, mediterranean, restaura..."
